# Hugging Face Whisper LoRA 전처리 실험 및 재개 학습

19GB 규모 Dataset Hub 음성을 스트리밍으로 읽어 전처리 조건을 바꿔가며 `openai/whisper-small` LoRA 학습을 실행합니다.

- HF 토큰은 `HF_TOKEN` 환경변수로만 전달합니다.
- 원본 음성과 transcript를 출력하거나 저장하지 않습니다.
- `checkpoint-*`가 있는 동일한 shard 출력 경로를 사용하면 중단된 학습을 재개합니다.
- 긴 오디오의 transcript가 전체 문장 하나뿐이면 잘못된 라벨을 만들지 않도록 청킹을 중단합니다.

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN 환경변수를 설정한 뒤 실행하세요.')

print('프로젝트:', PROJECT_ROOT)

## 1. Dataset Hub와 실험 설정

이 셀의 값만 바꿔서 실험 조건을 비교합니다. `DATASET_ID`, 컬럼명, source 값은 실제 Hub 데이터셋에 맞게 수정해야 합니다.

In [ ]:
# Dataset Hub
DATASET_ID = 'your-org/your-dataset'
DATASET_CONFIG = None
TRAIN_SPLIT = 'train'
EVAL_SPLIT = None  # 예: 'validation'; None이면 평가를 생략합니다.
AUDIO_COLUMN = 'audio'
TEXT_COLUMN = 'text'
SOURCE_COLUMN = 'source'
AIHUB_SOURCE_VALUES = {'aihub'}
TRANSCRIPT_MODE = 'full'  # 'full': 이미 청크/정렬됨, 'segments': timestamp segment 사용
SEGMENTS_COLUMN = 'segments'

# 재현성과 Dataset Hub shard
PREPROCESS_SEED = 42
NUM_SHARDS = 20
SHARD_INDEX = 0

# 오디오 기본 처리
TARGET_SAMPLE_RATE = 16_000
FORCE_MONO = True

# 속도 0.9~1.1배 랜덤 변경
SPEED_ENABLED = True
SPEED_MIN = 0.9
SPEED_MAX = 1.1
SPEED_PROBABILITY = 1.0

# RMS 음량 정규화 후 랜덤 gain 추가
RMS_NORMALIZE_ENABLED = True
TARGET_RMS_DB = -20.0
GAIN_ENABLED = True
GAIN_DB_MIN = -3.0
GAIN_DB_MAX = 3.0
GAIN_PROBABILITY = 1.0

# 일부 데이터에만 노이즈 추가
NOISE_ENABLED = True
NOISE_PROBABILITY = 0.2
NOISE_SNR_DB_MIN = 15.0
NOISE_SNR_DB_MAX = 30.0

# AIHub 데이터 일부에만 beep 추가
BEEP_ENABLED = True
BEEP_DATASET_ONLY = True
BEEP_PROBABILITY = 0.1
BEEP_DURATION_MS = 300
BEEP_FREQUENCY_HZ = 1000.0
BEEP_GAIN_DB = -6.0
BEEP_SOURCE_PATH = None  # 필요하면 로컬 beep.wav 절대경로 지정; None이면 tone 생성

# 20~30초 청킹과 overlap 10%
CHUNK_ENABLED = True
CHUNK_MIN_SECONDS = 20.0
CHUNK_MAX_SECONDS = 30.0
CHUNK_OVERLAP_RATIO = 0.10
RANDOMIZE_CHUNK_LENGTH = False

# 출력
OUTPUT_ROOT = PROJECT_ROOT / 'models' / 'whisper-lora-hf'
SHARD_OUTPUT = OUTPUT_ROOT / f'shard-{SHARD_INDEX:03d}-of-{NUM_SHARDS:03d}'

assert 0 <= SHARD_INDEX < NUM_SHARDS
assert 0 <= SPEED_MIN <= SPEED_MAX
assert 0 <= SPEED_PROBABILITY <= 1
assert 0 <= NOISE_PROBABILITY <= 1
assert 0 <= BEEP_PROBABILITY <= 1
assert 0 <= CHUNK_OVERLAP_RATIO < 1
assert 0 < CHUNK_MIN_SECONDS <= CHUNK_MAX_SECONDS

## 2. 학습 및 LoRA 하이퍼파라미터

streaming 데이터셋은 전체 길이를 즉시 알 수 없으므로 `TRAINING_MODE='steps'`를 기본으로 합니다. epoch 기반이 필요한 경우 `TRAINING_MODE='epochs'`로 바꿀 수 있습니다.

In [ ]:
MODEL_NAME = 'openai/whisper-small'
LANGUAGE = 'ko'
TASK = 'transcribe'

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q_proj', 'v_proj']

TRAINING_MODE = 'steps'  # 'steps' 또는 'epochs'
MAX_STEPS = 10_000
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.0
WARMUP_STEPS = 500
LR_SCHEDULER_TYPE = 'linear'
OPTIMIZER = 'adamw_torch'
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPSILON = 1e-8
MAX_GRAD_NORM = 1.0
FP16 = True
BF16 = False
GRADIENT_CHECKPOINTING = True
LOGGING_STEPS = 10
SAVE_STEPS = 500
EVAL_STEPS = 500
SAVE_TOTAL_LIMIT = 2
DATALOADER_NUM_WORKERS = 0
RESUME_CHECKPOINT = None  # None이면 SHARD_OUTPUT의 마지막 checkpoint 자동 선택

if TRAINING_MODE not in {'steps', 'epochs'}:
    raise ValueError("TRAINING_MODE는 'steps' 또는 'epochs'여야 합니다.")
if TRAINING_MODE == 'steps' and MAX_STEPS <= 0:
    raise ValueError('steps 모드에서는 MAX_STEPS가 양수여야 합니다.')
if TRAINING_MODE == 'epochs' and NUM_TRAIN_EPOCHS <= 0:
    raise ValueError('epochs 모드에서는 NUM_TRAIN_EPOCHS가 양수여야 합니다.')
if FP16 and BF16:
    raise ValueError('FP16과 BF16을 동시에 사용할 수 없습니다.')

In [ ]:
from datasets import load_dataset

dataset_kwargs = {
    'path': DATASET_ID,
    'split': TRAIN_SPLIT,
    'streaming': True,
    'token': os.environ['HF_TOKEN'],
}
if DATASET_CONFIG:
    dataset_kwargs['name'] = DATASET_CONFIG

def load_stream(split):
    kwargs = dict(dataset_kwargs, split=split)
    stream = load_dataset(**kwargs)
    return stream.shard(num_shards=NUM_SHARDS, index=SHARD_INDEX)

train_stream = load_stream(TRAIN_SPLIT)
print(train_stream)
print('컬럼:', list(train_stream.features or {}))

In [ ]:
sample = next(iter(train_stream))
print('컬럼:', sorted(sample.keys()))
print('오디오 타입:', type(sample[AUDIO_COLUMN]).__name__)
print('텍스트 길이:', len(str(sample[TEXT_COLUMN])))
if BEEP_ENABLED and BEEP_DATASET_ONLY and SOURCE_COLUMN not in sample:
    raise KeyError(f'BEEP_DATASET_ONLY=True인데 {SOURCE_COLUMN!r} 컬럼이 없습니다.')
if CHUNK_ENABLED and TRANSCRIPT_MODE == 'segments' and SEGMENTS_COLUMN not in sample:
    raise KeyError(f'TRANSCRIPT_MODE=segments인데 {SEGMENTS_COLUMN!r} 컬럼이 없습니다.')

## 3. 오디오 전처리 함수

각 row ID와 seed를 조합해 랜덤값을 결정하므로, 같은 shard를 checkpoint에서 재개해도 전처리 조건이 바뀌지 않습니다.

In [ ]:
import hashlib
import numpy as np

def rng_for(identifier):
    digest = hashlib.sha256(f'{PREPROCESS_SEED}:{identifier}'.encode()).digest()
    return np.random.default_rng(int.from_bytes(digest[:8], 'little'))

def as_mono_float32(audio):
    if isinstance(audio, dict):
        array = audio.get('array')
        sampling_rate = audio.get('sampling_rate')
        if array is None and audio.get('bytes') is not None:
            import io
            import soundfile as sf
            array, sampling_rate = sf.read(io.BytesIO(audio['bytes']), dtype='float32')
    else:
        array, sampling_rate = audio, TARGET_SAMPLE_RATE
    if array is None or sampling_rate is None:
        raise ValueError('오디오에 array와 sampling_rate가 필요합니다.')
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2 and FORCE_MONO:
        array = array.mean(axis=0 if array.shape[0] <= 8 else 1)
    if array.ndim != 1:
        raise ValueError(f'지원하지 않는 오디오 shape: {array.shape}')
    if int(sampling_rate) != TARGET_SAMPLE_RATE:
        old_x = np.linspace(0.0, 1.0, num=len(array), endpoint=False)
        new_length = max(1, round(len(array) * TARGET_SAMPLE_RATE / int(sampling_rate)))
        new_x = np.linspace(0.0, 1.0, num=new_length, endpoint=False)
        array = np.interp(new_x, old_x, array).astype(np.float32)
    return np.nan_to_num(array).astype(np.float32, copy=False)

def rms_db(audio):
    rms = float(np.sqrt(np.mean(np.square(audio), dtype=np.float64)))
    return 20.0 * math.log10(max(rms, 1e-8))

def speed_change(audio, factor):
    if abs(factor - 1.0) < 1e-6:
        return audio
    positions = np.arange(0, len(audio), factor, dtype=np.float64)
    return np.interp(positions, np.arange(len(audio)), audio).astype(np.float32)

def add_noise(audio, rng, snr_db):
    signal_rms = max(float(np.sqrt(np.mean(np.square(audio)))), 1e-8)
    noise_rms = signal_rms / (10.0 ** (snr_db / 20.0))
    return audio + rng.normal(0.0, noise_rms, size=len(audio)).astype(np.float32)

def make_beep():
    if BEEP_SOURCE_PATH:
        import soundfile as sf
        beep, sample_rate = sf.read(BEEP_SOURCE_PATH, dtype='float32')
        beep = as_mono_float32({'array': beep, 'sampling_rate': sample_rate})
        return beep
    length = max(1, round(TARGET_SAMPLE_RATE * BEEP_DURATION_MS / 1000.0))
    t = np.arange(length, dtype=np.float32) / TARGET_SAMPLE_RATE
    tone = np.sin(2.0 * np.pi * BEEP_FREQUENCY_HZ * t)
    return (tone * (10.0 ** (BEEP_GAIN_DB / 20.0))).astype(np.float32)

def add_beep(audio, rng):
    beep = make_beep()
    if len(audio) <= len(beep):
        start = 0
    else:
        start = int(rng.integers(0, len(audio) - len(beep)))
    output = audio.copy()
    output[start:start + len(beep)] = np.clip(output[start:start + len(beep)] + beep, -1.0, 1.0)
    return output

def apply_augmentations(audio, identifier, is_aihub):
    rng = rng_for(identifier)
    output = audio
    if SPEED_ENABLED and rng.random() < SPEED_PROBABILITY:
        output = speed_change(output, float(rng.uniform(SPEED_MIN, SPEED_MAX)))
    if RMS_NORMALIZE_ENABLED:
        output = output * (10.0 ** ((TARGET_RMS_DB - rms_db(output)) / 20.0))
    if GAIN_ENABLED and rng.random() < GAIN_PROBABILITY:
        output = output * (10.0 ** (float(rng.uniform(GAIN_DB_MIN, GAIN_DB_MAX)) / 20.0))
    if NOISE_ENABLED and rng.random() < NOISE_PROBABILITY:
        output = add_noise(output, rng, float(rng.uniform(NOISE_SNR_DB_MIN, NOISE_SNR_DB_MAX)))
    if BEEP_ENABLED and (not BEEP_DATASET_ONLY or is_aihub) and rng.random() < BEEP_PROBABILITY:
        output = add_beep(output, rng)
    return np.clip(output, -1.0, 1.0).astype(np.float32, copy=False)

In [ ]:
def segments_for_chunk(segments, start, end):
    texts = []
    for segment in segments:
        seg_start = float(segment.get('start', 0.0))
        seg_end = float(segment.get('end', seg_start))
        if seg_end > start and seg_start < end:
            text = str(segment.get('text', '')).strip()
            if text:
                texts.append(text)
    return ' '.join(texts).strip()

def make_chunks(audio, transcript, segments, identifier):
    duration = len(audio) / TARGET_SAMPLE_RATE
    if not CHUNK_ENABLED or duration <= CHUNK_MAX_SECONDS:
        return [(audio, transcript, identifier)]
    if TRANSCRIPT_MODE != 'segments' or not segments:
        raise ValueError('긴 오디오는 timestamp segments가 있어야 청킹할 수 있습니다.')
    results = []
    start = 0.0
    index = 0
    while start < duration:
        rng = rng_for(f'{identifier}:chunk:{index}')
        length = (float(rng.uniform(CHUNK_MIN_SECONDS, CHUNK_MAX_SECONDS))
                  if RANDOMIZE_CHUNK_LENGTH else CHUNK_MAX_SECONDS)
        end = min(duration, start + length)
        left = round(start * TARGET_SAMPLE_RATE)
        right = round(end * TARGET_SAMPLE_RATE)
        chunk_text = segments_for_chunk(segments, start, end)
        if chunk_text:
            results.append((audio[left:right], chunk_text, f'{identifier}__chunk_{index:04d}'))
        if end >= duration:
            break
        start = end - length * CHUNK_OVERLAP_RATIO
        index += 1
    return results

In [ ]:
def prepare_row(row, row_index):
    identifier = str(row.get('id', row.get('audio_id', row_index)))
    audio = as_mono_float32(row[AUDIO_COLUMN])
    source = str(row.get(SOURCE_COLUMN, '')).casefold()
    is_aihub = source in {value.casefold() for value in AIHUB_SOURCE_VALUES}
    audio = apply_augmentations(audio, identifier, is_aihub)
    transcript = str(row[TEXT_COLUMN]).strip()
    segments = row.get(SEGMENTS_COLUMN) if TRANSCRIPT_MODE == 'segments' else None
    return make_chunks(audio, transcript, segments, identifier)

class PreparedStreamingDataset:
    def __init__(self, source_dataset):
        self.source_dataset = source_dataset
    def __iter__(self):
        for row_index, row in enumerate(self.source_dataset):
            for audio, transcript, identifier in prepare_row(row, row_index):
                from tools.train_whisper_lora import prepare_features
                prepared = prepare_features(
                    {'audio': {'array': audio, 'sampling_rate': TARGET_SAMPLE_RATE},
                     'transcript': transcript},
                    processor,
                )
                prepared['identifier'] = identifier
                yield prepared

print('전처리 함수 준비 완료')

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES, bias='none', task_type=None,
))
# faster-whisper 배포를 위한 학습은 Transformers Whisper에서 수행합니다.
# encoder 본체와 encoder에 삽입된 LoRA adapter를 모두 고정하고 decoder만 학습합니다.
for name, parameter in model.named_parameters():
    if '.encoder.' in name or name.startswith('encoder.'):
        parameter.requires_grad = False
model.print_trainable_parameters()

prepared_train = PreparedStreamingDataset(train_stream)
prepared_eval = None
if EVAL_SPLIT:
    eval_stream = load_stream(EVAL_SPLIT)
    prepared_eval = PreparedStreamingDataset(eval_stream)
preview = next(iter(prepared_train))
print('feature shape:', preview['input_features'].shape)
print('label token count:', len(preview['labels']))

In [ ]:
import torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from tools.train_whisper_lora import WhisperDataCollator

SHARD_OUTPUT.mkdir(parents=True, exist_ok=True)
last_checkpoint = get_last_checkpoint(str(SHARD_OUTPUT))
resume_checkpoint = RESUME_CHECKPOINT or last_checkpoint

training_values = dict(
    output_dir=str(SHARD_OUTPUT),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS, lr_scheduler_type=LR_SCHEDULER_TYPE,
    optim=OPTIMIZER, adam_beta1=ADAM_BETA1, adam_beta2=ADAM_BETA2,
    adam_epsilon=ADAM_EPSILON, max_grad_norm=MAX_GRAD_NORM,
    fp16=FP16, bf16=BF16, gradient_checkpointing=GRADIENT_CHECKPOINTING,
    logging_steps=LOGGING_STEPS, save_steps=SAVE_STEPS, eval_steps=EVAL_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT, dataloader_num_workers=DATALOADER_NUM_WORKERS,
    remove_unused_columns=False, report_to=[], predict_with_generate=False,
    save_strategy='steps', evaluation_strategy='steps' if EVAL_SPLIT else 'no',
)
if TRAINING_MODE == 'steps':
    training_values['max_steps'] = MAX_STEPS
    training_values['num_train_epochs'] = 1
else:
    training_values['max_steps'] = -1
    training_values['num_train_epochs'] = NUM_TRAIN_EPOCHS
try:
    training_args = Seq2SeqTrainingArguments(**training_values)
except TypeError:
    training_values['eval_strategy'] = training_values.pop('evaluation_strategy')
    training_args = Seq2SeqTrainingArguments(**training_values)

trainer = Seq2SeqTrainer(
    args=training_args, model=model, train_dataset=prepared_train,
    eval_dataset=prepared_eval,
    data_collator=WhisperDataCollator(processor),
)
print('resume checkpoint:', resume_checkpoint)

In [ ]:
trainer.train(resume_from_checkpoint=resume_checkpoint)
trainer.save_model(SHARD_OUTPUT)
processor.save_pretrained(SHARD_OUTPUT)
(SHARD_OUTPUT / 'preprocessing_config.json').write_text(
    json.dumps({
        'dataset_id': DATASET_ID, 'train_split': TRAIN_SPLIT,
        'shard_index': SHARD_INDEX, 'num_shards': NUM_SHARDS,
        'preprocess_seed': PREPROCESS_SEED, 'transcript_mode': TRANSCRIPT_MODE,
        'speed': [SPEED_MIN, SPEED_MAX, SPEED_PROBABILITY],
        'target_rms_db': TARGET_RMS_DB, 'gain_db': [GAIN_DB_MIN, GAIN_DB_MAX],
        'noise_probability': NOISE_PROBABILITY, 'beep_probability': BEEP_PROBABILITY,
        'chunk_seconds': [CHUNK_MIN_SECONDS, CHUNK_MAX_SECONDS],
        'chunk_overlap_ratio': CHUNK_OVERLAP_RATIO,
        'training_mode': TRAINING_MODE, 'max_steps': MAX_STEPS,
        'num_train_epochs': NUM_TRAIN_EPOCHS, 'learning_rate': LEARNING_RATE,
        'batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    }, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
(SHARD_OUTPUT / '_SUCCESS').write_text('completed\n', encoding='utf-8')
print(f'shard {SHARD_INDEX} 학습 완료: {SHARD_OUTPUT}')